# 9 · Grounding  (per-article)

Verifies that every asserted **n-ary node** and **relation** is actually supported by the source
article, and drops (flags) the ones that are not. This is the factuality gate: schema validation
asks *"is this well-formed?"*, grounding asks *"does the text really say this?"*

Two checks, cheapest first:

1. **Evidence check (deterministic, free).** Does each `evidence` string actually occur in the
   article text? A quote that is not in the article was paraphrased or fabricated — a hallucination
   signal on its own, and it costs nothing to test. Normalised match (whitespace, quote characters,
   casefold) with a fuzzy fallback so lightly-reworded quotes are flagged `partial`, not failed.
2. **Support check (Gemini judge).** For the survivors, verbalise the assertion and ask whether the
   cited evidence supports it. Failures are classified **intrinsic** (evidence contradicts the
   assertion) vs **extrinsic** (evidence does not mention it — injected outside knowledge).

**Nodes are grounded before relations.** A node makes its own claim (this event exists, with this
type) that no relation tests, and if a node is dropped its edges must go with it — so grounding
nodes first also saves judge calls on relations that are about to be discarded anyway.

Nothing is deleted: every item keeps `grounding.*` fields so drop-rate and hallucination-rate stay
measurable and inspectable.

Reads  `results/<id>/e2e/nary_graph_dedup_extended_consolidated.json` + `dataset/articles/<id>.txt`
Writes `results/<id>/e2e/nary_graph_grounded.json`

In [70]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [71]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

ARTICLE_ID   = "art5"                      # single-article run; batch cell at the end loops all
GEMINI_MODEL = "gemini-2.5-flash"          # the grounding judge

GRAPH_NAME   = "nary_graph_dedup_extended_consolidated.json"   # input graph (per article, e2e/)
OUT_NAME     = "nary_graph_grounded.json"                      # output written next to it
TRACK_DIR    = "e2e"                                           # results/<id>/<TRACK_DIR>/

# Evidence check knobs
FUZZY_THRESHOLD = 0.85     # difflib ratio; >= this against the best window -> "partial", else "missing"
JUDGE_PARTIAL   = True     # send `partial` evidence to the judge too (it decides on the text it has)
MAX_SPAN_CHARS      = 400  # elided quote whose fragments span more than this = not one quote -> missing
MAX_SENTENCE_CHARS  = 600  # cap on sentence-expanded evidence (pathological paragraphs)

# Judge knobs
JUDGE_BATCH = 25           # assertions per Gemini call (keeps prompts readable, retries cheap)
JUDGE_RETRIES = 3          # attempts per batch before giving up (transient API/JSON errors)
JUDGE_RETRY_BACKOFF = 5    # seconds between retries; linear backoff = BACKOFF * attempt

ARTICLE_DIR = ROOT / "dataset" / "articles"
print("article:", ARTICLE_ID, "| gemini:", GEMINI_MODEL, "| root:", ROOT)

article: art5 | gemini: gemini-2.5-flash | root: /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction


In [72]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    last_latency_s = time.perf_counter() - t0
    return json.loads(resp.text)

Note: you may need to restart the kernel to use updated packages.


## 1. Load the graph and the source article

In [73]:
def load_graph(article_id):
    path = ROOT / "results" / article_id / TRACK_DIR / GRAPH_NAME
    return json.loads(path.read_text()), path

def load_article(article_id):
    return (ARTICLE_DIR / f"{article_id}.txt").read_text()

graph, graph_path = load_graph(ARTICLE_ID)
article_text = load_article(ARTICLE_ID)

print(f"{graph_path.relative_to(ROOT)}")
print(f"  entities   {len(graph.get('entities', []))}")
print(f"  nary_nodes {len(graph.get('nary_nodes', []))}")
print(f"  relations  {len(graph.get('relations', []))}")
print(f"  article    {len(article_text):,} chars")

results/art5/e2e/nary_graph_dedup_extended_consolidated.json
  entities   58
  nary_nodes 37
  relations  164
  article    4,579 chars


## 2. Evidence check — is the quoted evidence really in the article?

Deterministic and free, so it runs before any Gemini call. Quotes are normalised (whitespace
collapsed, curly quotes and dashes unified, casefolded) before matching, because the extractor
often re-types a quote with different punctuation. An exact normalised substring hit is `found`;
otherwise the best same-length window is scored with `difflib` and counts as `partial` above
`FUZZY_THRESHOLD` (lightly reworded, still traceable) or `missing` below it.

`missing` evidence is the strongest cheap hallucination signal you have — the model cited text the
article does not contain.

Two real failure modes this handles, both found on the current articles:

* **Elided quotes** — the extractor compresses a long quote (`"Belgium recorded ... 39% more than
  usual"`). Whole-string matching can never hit those, so the quote is split on the ellipsis and the
  fragments are matched in order; faithful-but-elided counts as `partial`.
* **Mojibake in the article file** — some scraped articles contain UTF-8-read-as-Latin-1 damage
  (`Â£3.74m` instead of `£3.74m`), which made every quote containing `£` look unmatched.
  `fix_mojibake()` repairs it at match time, but the real fix is to re-scrape those articles.

**Matched quotes are resolved back to character offsets in the article**, and widened to sentence
boundaries. The recovered sentence — the article's own words — is what the judge reads in step 3,
not the extractor's retyped quote. That closes a real loophole: an elided quote lets the model
choose what the judge sees, and the elided middle is exactly where an inconvenient number or
qualifier tends to sit (`"Belgium recorded ... 39% more than usual"` hides *1,222 excess deaths*;
`"...risk was \"very high\""` hides *compared to summer norms*). Offsets also give you exact,
article-anchored provenance instead of a free-floating string.

In [74]:
import re, unicodedata
from difflib import SequenceMatcher

_QUOTES = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
                         "\u2013": "-", "\u2014": "-", "\u2026": "..."})

def fix_mojibake(s: str) -> str:
    """Undo UTF-8-read-as-Latin-1 damage ("\u00c2\u00a3" -> "\u00a3") seen in some scraped article files.
    Cheap and safe: only re-decodes when the tell-tale \u00c2/\u00e2 sequences are present."""
    if not s or ("\u00c2" not in s and "\u00e3" not in s and "\u00e2" not in s):
        return s
    try:
        return s.encode("latin-1", "ignore").decode("utf-8", "ignore")
    except Exception:
        return s

def norm_text(s: str) -> str:
    """Normalise for matching: repair mojibake, unify quotes/dashes, collapse space, casefold."""
    s = fix_mojibake(s or "")
    s = unicodedata.normalize("NFKC", s).translate(_QUOTES)
    return re.sub(r"\s+", " ", s).strip().casefold()

def build_index(raw: str):
    """Normalise the article while keeping a map from every normalised char back to its RAW offset.
    That map is what lets a matched quote be resolved to real character offsets in the article."""
    raw = fix_mojibake(raw)
    chars, idx, prev_space = [], [], False
    for i, ch in enumerate(unicodedata.normalize("NFKC", raw).translate(_QUOTES)):
        if ch.isspace():
            if prev_space or not chars:
                continue
            chars.append(" "); idx.append(i); prev_space = True
        else:
            chars.append(ch.casefold()); idx.append(i); prev_space = False
    return "".join(chars), idx, raw

# The extractor sometimes elides the middle of a long quote ("Belgium recorded ... 39% more than
# usual"). Rather than failing those, split on the ellipsis and locate the fragments in order --
# the span from the first to the last fragment IS the real quote, elided middle included.
_ELLIPSIS_RE = re.compile(r"\s*(?:\.\.\.|\u2026)\s*")

def locate_quote(qn: str, art_norm: str, idx_map):
    """Find one normalised quote in the article. -> (status, ratio, (start, end) | None) in RAW offsets."""
    if not qn:
        return "missing", 0.0, None
    if qn in art_norm:
        s = art_norm.index(qn)
        return "found", 1.0, (idx_map[s], idx_map[s + len(qn) - 1] + 1)

    frags = [f for f in _ELLIPSIS_RE.split(qn) if len(f) > 3]
    if len(frags) > 1:                                   # elided quote: walk the fragments in order
        pos, first, last = 0, None, None
        for f in frags:
            i = art_norm.find(f, pos)
            if i < 0:
                first = None; break
            if first is None:
                first = i
            last = i + len(f); pos = last
        if first is not None:
            span = (idx_map[first], idx_map[last - 1] + 1)
            if span[1] - span[0] <= MAX_SPAN_CHARS:      # guard: fragments matched far apart
                return "partial", 1.0, span
            return "missing", 0.0, None

    r = best_window_ratio(qn, art_norm)
    return ("partial", r, None) if r >= FUZZY_THRESHOLD else ("missing", r, None)

def best_window_ratio(needle: str, haystack: str) -> float:
    """Best difflib ratio of `needle` against any same-length window of `haystack` (coarse stride)."""
    n = len(needle)
    if n == 0 or len(haystack) < n:
        return 0.0
    stride = max(1, n // 4)
    best = 0.0
    for i in range(0, len(haystack) - n + 1, stride):
        r = SequenceMatcher(None, needle, haystack[i:i + n]).ratio()
        if r > best:
            best = r
            if best >= 0.99:
                break
    return best

_SENT_END = re.compile(r"[.!?]")

def expand_to_sentences(raw: str, start: int, end: int):
    """Widen a span outward to sentence boundaries, so the judge sees the qualifying clauses the
    extractor may have cut ("...very high" -> "...very high compared to summer norms")."""
    left = raw.rfind(".", 0, start)
    left = 0 if left < 0 else left + 1
    m = _SENT_END.search(raw, end)
    right = len(raw) if not m else m.end()
    if right - left > MAX_SENTENCE_CHARS:                # pathological paragraph: keep the raw span
        return raw[start:end].strip()
    return raw[left:right].strip()

def flatten_evidence(ev):
    """Evidence is a str, list[str], or list[list[str]] (relations). -> flat list[str]."""
    out = []
    if ev is None:
        return out
    if isinstance(ev, str):
        return [ev]
    for item in ev:
        if isinstance(item, str):
            out.append(item)
        elif isinstance(item, list):
            out.extend(x for x in item if isinstance(x, str))
    return out

def check_evidence(evidence, art_norm, idx_map, raw):
    """-> (status, detail, resolved). `resolved` carries, per quote, its RAW offsets and the
    recovered sentence(s) -- the article's own words, not the extractor's retyped quote."""
    quotes = flatten_evidence(evidence)
    if not quotes:
        return "none", {"n_quotes": 0, "found": 0, "partial": 0, "missing": 0, "worst_ratio": None}, []

    tally = {"found": 0, "partial": 0, "missing": 0}
    worst, resolved = 1.0, []
    for q in quotes:
        st, r, span = locate_quote(norm_text(q), art_norm, idx_map)
        tally[st] += 1
        worst = min(worst, r)
        item = {"quote": q, "status": st, "ratio": round(r, 3), "start": None, "end": None,
                "matched_text": None, "sentence": None}
        if span:
            item.update({"start": span[0], "end": span[1],
                         "matched_text": raw[span[0]:span[1]].strip(),
                         "sentence": expand_to_sentences(raw, *span)})
        resolved.append(item)

    status = ("found"   if tally["missing"] == 0 and tally["partial"] == 0 else
              "partial" if tally["missing"] == 0 else "missing")
    return status, {"n_quotes": len(quotes), **tally, "worst_ratio": round(worst, 3)}, resolved

def judge_text(item):
    """What the judge should read: the RECOVERED article sentences when we resolved the quote,
    falling back to the extractor's own quote only when we could not locate it.
    This matters -- an elided quote lets the model choose what the judge sees, and the elided middle
    is exactly where an inconvenient number or qualifier tends to sit."""
    res = item.get("grounding", {}).get("evidence_resolved") or []
    out, seen = [], set()
    for r in res:
        t = r.get("sentence") or r.get("matched_text") or r.get("quote")
        if t and t not in seen:
            seen.add(t); out.append(t)
    return out or flatten_evidence(item.get("evidence"))

ARTICLE_NORM, ARTICLE_IDX, ARTICLE_RAW = build_index(article_text)
print("article normalised:", len(ARTICLE_NORM), "chars | index map:", len(ARTICLE_IDX))

article normalised: 4578 chars | index map: 4578


In [75]:
# Run the evidence check over nodes and relations (no API calls).
def evidence_pass(graph, art_norm, idx_map, raw):
    for kind in ("nary_nodes", "relations"):
        for item in graph.get(kind, []):
            st, detail, resolved = check_evidence(item.get("evidence"), art_norm, idx_map, raw)
            item.setdefault("grounding", {}).update(
                {"evidence_status": st, "evidence_detail": detail, "evidence_resolved": resolved})
    return graph

graph = evidence_pass(graph, ARTICLE_NORM, ARTICLE_IDX, ARTICLE_RAW)

from collections import Counter
for kind in ("nary_nodes", "relations"):
    c = Counter(x["grounding"]["evidence_status"] for x in graph.get(kind, []))
    print(f"{kind:11} " + "  ".join(f"{k}={c[k]}" for k in ("found", "partial", "missing", "none") if c[k]))

n_res = sum(1 for kind in ("nary_nodes", "relations") for x in graph.get(kind, [])
            for r in x["grounding"]["evidence_resolved"] if r.get("start") is not None)
n_all = sum(len(x["grounding"]["evidence_resolved"]) for kind in ("nary_nodes", "relations")
            for x in graph.get(kind, []))
print(f"\nquotes resolved to article offsets: {n_res}/{n_all}")

# Elided quotes are where recovery pays off: the middle the extractor cut out is restored.
print("\nrecovered elisions (extractor quote -> article's own sentence):")
shown = 0
for kind in ("nary_nodes", "relations"):
    for x in graph.get(kind, []):
        for r in x["grounding"]["evidence_resolved"]:
            if r["status"] == "partial" and r.get("sentence") and shown < 5:
                print(f'  quote    : {r["quote"][:95]}')
                print(f'  recovered: {r["sentence"][:150]}\n')
                shown += 1

bad = [(x.get("id") or f'{x.get("subject_text")} -{x.get("property")}-> {x.get("object_text")}',
        x["grounding"]["evidence_detail"])
       for kind in ("nary_nodes", "relations") for x in graph.get(kind, [])
       if x["grounding"]["evidence_status"] == "missing"]
print(f"{len(bad)} item(s) with missing evidence:")
for label, d in bad[:10]:
    print("  -", str(label)[:80], "|", d)

nary_nodes  found=37
relations   found=146  partial=2  none=16

quotes resolved to article offsets: 183/185

recovered elisions (extractor quote -> article's own sentence):
0 item(s) with missing evidence:


## 3. Support check — does the evidence actually assert this?

Only items whose evidence survived step 2 are judged (`missing` is already a failure; `partial`
optionally goes through, controlled by `JUDGE_PARTIAL`).

The judge sees the **recovered article sentences** from step 2 (falling back to the extractor's
quote only where the span could not be located). Each assertion is **verbalised** first — `(EV1, location, E1)` means nothing to a judge, but *"The
event 'France records excess deaths' is located in France"* does. The judge sees **only the cited
evidence**, never the whole article: that is the point of carrying spans, it keeps the judgment
focused and stops the judge from finding support elsewhere that the extractor never used.

Failures are classified so the two error modes stay separable in the metrics:

* **intrinsic** — the evidence contradicts or misstates the assertion (a misreading).
* **extrinsic** — the evidence does not mention it; the model supplied outside knowledge (a leak).

In [76]:
GROUNDING_SYSTEM_PROMPT = """You are a strict grounding judge for a knowledge-graph extraction pipeline. You are given a batch of assertions extracted from a news article. Each assertion comes with the exact evidence quotes the extractor cited from that article.

For each assertion decide whether THE CITED EVIDENCE SUPPORTS IT. Judge only against the evidence shown. Do not use outside knowledge, and do not assume anything the evidence does not state, even if you believe it is true in the world.

Answer for each assertion:
- "supported": true if a careful reader of the evidence would agree the evidence asserts (or directly entails) the assertion; false otherwise.
- "failure_type": null when supported. When not supported, exactly one of:
    "intrinsic"  - the evidence contradicts or misstates the assertion (wrong number, wrong entity, wrong direction, overstated certainty).
    "extrinsic"  - the evidence simply does not mention the assertion; it may be true in the world but the cited text does not say it.
- "note": one short clause naming the reason. Be specific ("evidence gives 2,025 not 2,000", "evidence never mentions the agency").

Be strict but not pedantic. Paraphrase is fine: the assertion does not need to repeat the evidence wording. Direction matters: if the evidence supports the reverse relation but not the stated one, that is "intrinsic". A vaguer assertion entailed by more specific evidence is supported; a more specific assertion than the evidence warrants is not.

Return JSON only: {"results": [{"index": <int>, "supported": <bool>, "failure_type": <string|null>, "note": "<string>"}]}. Return exactly one result per assertion, echoing its index."""

def verbalise_node(node):
    kind = node.get("node_kind") or "Node"
    return (f'A {kind.lower()} of type "{node.get("type")}" exists, described as '
            f'"{node.get("label")}".')

def verbalise_relation(rel):
    prop = rel.get("property")
    return (f'"{rel.get("subject_text")}" ({rel.get("subject_type")}) '
            f'--[{prop}]--> "{rel.get("object_text")}" ({rel.get("object_type")}).')

def judge_batch(items, verbalise):
    """items: list of dicts with .evidence -> attach grounding verdicts in place."""
    payload = []
    for i, it in enumerate(items):
        payload.append({"index": i,
                        "assertion": verbalise(it),
                        "evidence": judge_text(it)})   # recovered article sentences, not the retyped quote
    msg = "Assertions to judge:\n" + json.dumps(payload, ensure_ascii=False, indent=1)
    out = gemini_json(GROUNDING_SYSTEM_PROMPT, msg)
    by_idx = {r.get("index"): r for r in out.get("results", []) if isinstance(r, dict)}
    for i, it in enumerate(items):
        r = by_idx.get(i)
        g = it.setdefault("grounding", {})
        if r is None:                       # judge skipped it -> keep, but mark unjudged
            g.update({"supported": None, "failure_type": None, "note": "judge returned no verdict"})
        else:
            g.update({"supported": bool(r.get("supported")),
                      "failure_type": r.get("failure_type"),
                      "note": (r.get("note") or "")[:200]})
    return items

def judgeable(item):
    st = item.get("grounding", {}).get("evidence_status")
    return st == "found" or (JUDGE_PARTIAL and st == "partial")

def run_judge(items, verbalise, label):
    todo = [it for it in items if judgeable(it)]
    skipped = len(items) - len(todo)
    for start in range(0, len(todo), JUDGE_BATCH):
        chunk = todo[start:start + JUDGE_BATCH]
        for attempt in range(1, JUDGE_RETRIES + 1):
            try:
                judge_batch(chunk, verbalise)
                break                               # success -> stop retrying
            except Exception as e:                  # transient API/JSON error -> back off and retry
                if attempt < JUDGE_RETRIES:
                    wait = JUDGE_RETRY_BACKOFF * attempt
                    print(f"  ! {label} batch {start}-{start+len(chunk)} attempt {attempt}/{JUDGE_RETRIES} "
                          f"failed: {type(e).__name__}: {e} — retrying in {wait:.0f}s")
                    time.sleep(wait)
                else:                               # exhausted retries: one bad batch must not kill the run
                    print(f"  ! {label} batch {start}-{start+len(chunk)} failed after {JUDGE_RETRIES} attempts: "
                          f"{type(e).__name__}: {e}")
                    for it in chunk:
                        it.setdefault("grounding", {}).update(
                            {"supported": None, "failure_type": None, "note": f"judge error: {type(e).__name__}"})
        print(f"  {label}: judged {min(start+JUDGE_BATCH, len(todo))}/{len(todo)}  ({last_latency_s:.1f}s)")
    # anything not sent to the judge fails on evidence alone
    for it in items:
        if not judgeable(it):
            it.setdefault("grounding", {}).setdefault("supported", False)
            it["grounding"].setdefault("failure_type", "extrinsic")
            it["grounding"].setdefault("note", "evidence not found in article")
    return items, skipped

In [77]:
# --- 3a. Ground the NODES first ----------------------------------------------------------
# A node asserts its own existence and type — nothing else in the graph tests that. And if a node
# fails, its relations must go too, so doing nodes first saves judging edges that are about to drop.
nodes = graph.get("nary_nodes", [])
print(f"grounding {len(nodes)} n-ary nodes")
run_judge(nodes, verbalise_node, "nodes")

for n in nodes:
    g = n["grounding"]
    g["grounded"] = bool(g.get("supported"))

kept = [n for n in nodes if n["grounding"]["grounded"]]
dropped_ids = {n["id"] for n in nodes if not n["grounding"]["grounded"]}
print(f"\nnodes grounded {len(kept)}/{len(nodes)}   dropped: {sorted(dropped_ids) if dropped_ids else 'none'}")
for n in nodes:
    if not n["grounding"]["grounded"]:
        print(f'  - {n["id"]:5} {str(n.get("label"))[:52]:54} {n["grounding"].get("failure_type")}: {n["grounding"].get("note","")[:70]}')

grounding 37 n-ary nodes
  nodes: judged 25/37  (20.3s)
  nodes: judged 37/37  (7.9s)

nodes grounded 37/37   dropped: none


In [78]:
# --- 3b. Ground the RELATIONS (skipping any whose endpoint node was dropped) --------------
rels = graph.get("relations", [])
for r in rels:
    g = r.setdefault("grounding", {})
    orphan = (r.get("subject_id") in dropped_ids) or (r.get("object_id") in dropped_ids)
    g["orphaned"] = orphan
    if orphan:   # endpoint gone -> the edge cannot stand; no judge call needed
        g.update({"supported": False, "failure_type": "orphan",
                  "note": "endpoint node failed grounding"})

live = [r for r in rels if not r["grounding"]["orphaned"]]
print(f"grounding {len(live)} relations ({len(rels)-len(live)} orphaned by dropped nodes)")
run_judge(live, verbalise_relation, "relations")

for r in rels:
    g = r["grounding"]
    g["grounded"] = bool(g.get("supported")) and not g["orphaned"]

kept_r = sum(r["grounding"]["grounded"] for r in rels)
print(f"\nrelations grounded {kept_r}/{len(rels)}")
for r in rels:
    if not r["grounding"]["grounded"]:
        print(f'  - {str(r.get("subject_text"))[:26]:28} -{r.get("property"):18}-> {str(r.get("object_text"))[:22]:24} '
              f'{r["grounding"].get("failure_type")}: {r["grounding"].get("note","")[:60]}')

grounding 164 relations (0 orphaned by dropped nodes)
  relations: judged 25/148  (20.4s)
  relations: judged 50/148  (19.9s)
  relations: judged 75/148  (17.3s)
  relations: judged 100/148  (18.9s)
  relations: judged 125/148  (17.8s)
  relations: judged 148/148  (20.3s)

relations grounded 103/164
  - Tens of thousands abandone   -about             -> Nigerian Richard Itoro   extrinsic: evidence states he is one of tens of thousands, not that the
  - Woman died in Ituri          -reportsOn         -> pregnant woman           extrinsic: evidence refers to 'The woman' not specifically 'pregnant wo
  - Confirmed Ebola cases in D   -location          -> Democratic Republic of   extrinsic: evidence does not mention confirmed Ebola cases
  - Suspected Ebola cases in D   -location          -> Democratic Republic of   extrinsic: evidence does not mention suspected Ebola cases
  - Ebola deaths in DR Congo     -location          -> Democratic Republic of   extrinsic: evidence does not mention 

## 4. Metrics and write-out

Grounding rate, plus the split that matters for the thesis: **intrinsic vs extrinsic** failures
(misreading vs. injected world knowledge) and grounding rate **by `type_source`** — do the minted
`vocab` types stand up as well as the schema.org ones? That is the highest-freedom part of the
output, so it is where hallucination should concentrate if it concentrates anywhere.

Nothing is removed from the file: `grounding.grounded` marks the verdict and the rest of the
`grounding` block records why, so drop rate stays auditable and you can re-threshold later.

In [79]:
def grounding_report(graph):
    rows = {}
    for kind in ("nary_nodes", "relations"):
        items = graph.get(kind, [])
        if not items: continue
        g = [x.get("grounding", {}) for x in items]
        n_ok  = sum(bool(x.get("grounded")) for x in g)
        fails = Counter(x.get("failure_type") for x in g if not x.get("grounded"))
        ev    = Counter(x.get("evidence_status") for x in g)
        rows[kind] = {"total": len(items), "grounded": n_ok,
                      "rate": round(n_ok / len(items), 3) if items else None,
                      "failures": dict(fails), "evidence": dict(ev)}
    # by type_source: schema.org terms vs minted vocab terms
    by_src = {}
    for n in graph.get("nary_nodes", []):
        src = n.get("type_source", "unknown")
        d = by_src.setdefault(src, {"total": 0, "grounded": 0})
        d["total"] += 1; d["grounded"] += bool(n.get("grounding", {}).get("grounded"))
    for src, d in by_src.items():
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
    rows["nodes_by_type_source"] = by_src
    return rows

report = grounding_report(graph)
print(json.dumps(report, indent=2))

{
  "nary_nodes": {
    "total": 37,
    "grounded": 37,
    "rate": 1.0,
    "failures": {},
    "evidence": {
      "found": 37
    }
  },
  "relations": {
    "total": 164,
    "grounded": 103,
    "rate": 0.628,
    "failures": {
      "extrinsic": 26,
      "intrinsic": 35
    },
    "evidence": {
      "found": 146,
      "partial": 2,
      "none": 16
    }
  },
  "nodes_by_type_source": {
    "vocab": {
      "total": 4,
      "grounded": 4,
      "rate": 1.0
    },
    "unknown": {
      "total": 33,
      "grounded": 33,
      "rate": 1.0
    }
  }
}


In [80]:
graph["grounding_report"] = report
graph["grounding_config"] = {"model": GEMINI_MODEL, "fuzzy_threshold": FUZZY_THRESHOLD,
                             "judge_partial": JUDGE_PARTIAL, "judge_batch": JUDGE_BATCH}

out_path = ROOT / "results" / ARTICLE_ID / TRACK_DIR / OUT_NAME
out_path.write_text(json.dumps(graph, ensure_ascii=False, indent=2))
print("wrote", out_path.relative_to(ROOT))

wrote results/art5/e2e/nary_graph_grounded.json


In [81]:
# --- Judgment log: a flat, reviewable record of EVERY grounding decision ------------------
# One row per node and per relation: what was judged (the verbalised assertion), the evidence the
# JUDGE actually saw (recovered article sentences), the evidence the extractor CITED, the decision,
# and its justification. Written next to the grounded graph as JSON (full) + CSV (skim in Excel).
import csv

JUDGMENTS_JSON = "grounding_judgments.json"
JUDGMENTS_CSV  = "grounding_judgments.csv"

def _decision(g):
    s = g.get("supported")
    return "supported" if s else ("unjudged" if s is None else "rejected")

def build_judgment_log(graph, article_id):
    rows = []
    for n in graph.get("nary_nodes", []):
        g = n.get("grounding", {})
        rows.append({
            "article_id": article_id, "kind": "node", "id": n.get("id"),
            "assertion": verbalise_node(n),
            "subject": None, "property": None, "object": None,
            "evidence_status": g.get("evidence_status"), "orphaned": None,
            "decision": _decision(g), "supported": g.get("supported"), "grounded": g.get("grounded"),
            "failure_type": g.get("failure_type"), "justification": g.get("note"),
            "evidence_judged": judge_text(n),                       # what the judge read
            "evidence_cited": flatten_evidence(n.get("evidence")),  # what the extractor quoted
        })
    for r in graph.get("relations", []):
        g = r.get("grounding", {})
        rows.append({
            "article_id": article_id, "kind": "relation",
            "id": f'{r.get("subject_id")}-{r.get("property")}-{r.get("object_id")}',
            "assertion": verbalise_relation(r),
            "subject": r.get("subject_text"), "property": r.get("property"), "object": r.get("object_text"),
            "evidence_status": g.get("evidence_status"), "orphaned": g.get("orphaned"),
            "decision": _decision(g), "supported": g.get("supported"), "grounded": g.get("grounded"),
            "failure_type": g.get("failure_type"), "justification": g.get("note"),
            "evidence_judged": judge_text(r),
            "evidence_cited": flatten_evidence(r.get("evidence")),
        })
    return rows

def write_judgment_log(graph, article_id):
    rows = build_judgment_log(graph, article_id)
    d = ROOT / "results" / article_id / TRACK_DIR
    (d / JUDGMENTS_JSON).write_text(json.dumps(rows, ensure_ascii=False, indent=2))
    cols = ["article_id","kind","id","assertion","subject","property","object",
            "evidence_status","orphaned","decision","supported","grounded","failure_type",
            "justification","evidence_judged","evidence_cited"]
    with (d / JUDGMENTS_CSV).open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
        for row in rows:                              # list-valued evidence -> one readable cell
            flat = dict(row)
            flat["evidence_judged"] = " || ".join(row["evidence_judged"] or [])
            flat["evidence_cited"]  = " || ".join(row["evidence_cited"] or [])
            w.writerow(flat)
    return rows

# write the log for the single-article run above
_jrows = write_judgment_log(graph, ARTICLE_ID)
_rej = [r for r in _jrows if r["decision"] == "rejected"]
print(f"wrote {(ROOT / 'results' / ARTICLE_ID / TRACK_DIR / JUDGMENTS_JSON).relative_to(ROOT)}"
      f"  (+ .csv)  — {len(_jrows)} judgments, {len(_rej)} rejected")
for r in _rej[:10]:
    print(f'  [{r["kind"]:8}] {r["assertion"][:70]}')
    print(f'             -> {r["failure_type"]}: {r["justification"]}')


wrote results/art5/e2e/grounding_judgments.json  (+ .csv)  — 201 judgments, 61 rejected
  [relation] "Tens of thousands abandoned lives" (Statement) --[about]--> "Nigerian
             -> extrinsic: evidence states he is one of tens of thousands, not that the statement is about him
  [relation] "Woman died in Ituri" (Report) --[reportsOn]--> "pregnant woman" (Pers
             -> extrinsic: evidence refers to 'The woman' not specifically 'pregnant woman'
  [relation] "Confirmed Ebola cases in DR Congo" (CaseCount) --[location]--> "Democ
             -> extrinsic: evidence does not mention confirmed Ebola cases
  [relation] "Suspected Ebola cases in DR Congo" (SuspectedCaseCount) --[location]-
             -> extrinsic: evidence does not mention suspected Ebola cases
  [relation] "Ebola deaths in DR Congo" (DeathCount) --[location]--> "Democratic Re
             -> extrinsic: evidence does not mention Ebola deaths
  [relation] "Public gatherings banned" (PublicGatheringBanEvent) --[loca

## 5. Validate the judge  ← do this before trusting any number above

The grounding rate is only as good as the judge producing it. Sample ~50 assertions, label them
yourself, and measure agreement. Until that is done the numbers are unvalidated.

The cell below writes a blank sheet with the verdict hidden; fill `human` with `y`/`n`, re-run the
scoring cell, and you get the agreement figure to report — plus the disagreements to read, which is
usually where you find the judge is too lenient on `extrinsic` cases.

In [67]:
import random

SAMPLE_N = 50
random.seed(0)

def build_sample(graph, n=SAMPLE_N):
    pool = ([{"kind": "node", "id": x.get("id"), "assertion": verbalise_node(x),
              "evidence": judge_text(x), "judge": x["grounding"].get("supported")}
             for x in graph.get("nary_nodes", []) if "grounding" in x] +
            [{"kind": "relation", "id": f'{x.get("subject_id")}-{x.get("property")}-{x.get("object_id")}',
              "assertion": verbalise_relation(x),
              "evidence": judge_text(x), "judge": x["grounding"].get("supported")}
             for x in graph.get("relations", []) if "grounding" in x])
    # stratify: keep both verdicts represented so leniency and strictness are both testable
    yes = [p for p in pool if p["judge"] is True]
    no  = [p for p in pool if p["judge"] is False]
    random.shuffle(yes); random.shuffle(no)
    half = n // 2
    sample = yes[:half] + no[:n - half]
    if len(sample) < n:                        # top up from whichever side has more
        rest = [p for p in yes[half:] + no[n-half:]][: n - len(sample)]
        sample += rest
    random.shuffle(sample)
    for i, s in enumerate(sample):
        s["index"] = i; s["human"] = ""        # <- fill in "y" / "n"
    return sample

sample = build_sample(graph)
sample_path = ROOT / "results" / ARTICLE_ID / TRACK_DIR / "grounding_judge_sample.json"
sample_path.write_text(json.dumps([{k: v for k, v in s.items() if k != "judge"} for s in sample],
                                  ensure_ascii=False, indent=2))
(ROOT / "results" / ARTICLE_ID / TRACK_DIR / "grounding_judge_key.json").write_text(
    json.dumps({str(s["index"]): s["judge"] for s in sample}, indent=2))
print("wrote", sample_path.relative_to(ROOT), f"({len(sample)} items) — fill in the `human` field, then run the next cell")

wrote results/art5/e2e/grounding_judge_sample.json (50 items) — fill in the `human` field, then run the next cell


In [68]:
# Score human labels against the judge (run after filling in `human` in the sample file).
def score_agreement(article_id=ARTICLE_ID):
    d = ROOT / "results" / article_id / TRACK_DIR
    labelled = json.loads((d / "grounding_judge_sample.json").read_text())
    key      = json.loads((d / "grounding_judge_key.json").read_text())
    pairs = [(str(x["index"]), x.get("human", "").strip().lower()) for x in labelled]
    pairs = [(i, h) for i, h in pairs if h in ("y", "n")]
    if not pairs:
        print("no human labels yet — fill the `human` field with y/n in grounding_judge_sample.json")
        return
    agree = sum((h == "y") == bool(key[i]) for i, h in pairs)
    print(f"labelled {len(pairs)}/{len(labelled)} | agreement {agree}/{len(pairs)} = {agree/len(pairs):.1%}")
    # where they disagree, and in which direction
    lenient = sum(h == "n" and bool(key[i]) for i, h in pairs)     # judge said supported, human said no
    strict  = sum(h == "y" and not bool(key[i]) for i, h in pairs) # judge said no, human said supported
    print(f"  judge too lenient: {lenient}   judge too strict: {strict}")
    by_id = {str(x["index"]): x for x in labelled}
    for i, h in pairs:
        if (h == "y") != bool(key[i]):
            x = by_id[i]
            print(f"\n  [{i}] judge={bool(key[i])} human={h}  {x['assertion'][:110]}")
            for q in x["evidence"][:2]:
                print(f"        ev: {q[:110]}")

score_agreement()

no human labels yet — fill the `human` field with y/n in grounding_judge_sample.json


## 6. Batch — ground every article

In [69]:
def ground_article(article_id, verbose=False):
    graph, _ = load_graph(article_id)
    art_norm, idx_map, raw = build_index(load_article(article_id))
    graph = evidence_pass(graph, art_norm, idx_map, raw)

    nodes = graph.get("nary_nodes", [])
    run_judge(nodes, verbalise_node, "nodes") if verbose else _quiet(run_judge, nodes, verbalise_node, "nodes")
    for n in nodes:
        n["grounding"]["grounded"] = bool(n["grounding"].get("supported"))
    dropped = {n["id"] for n in nodes if not n["grounding"]["grounded"]}

    rels = graph.get("relations", [])
    for r in rels:
        g = r.setdefault("grounding", {})
        g["orphaned"] = (r.get("subject_id") in dropped) or (r.get("object_id") in dropped)
        if g["orphaned"]:
            g.update({"supported": False, "failure_type": "orphan", "note": "endpoint node failed grounding"})
    live = [r for r in rels if not r["grounding"]["orphaned"]]
    run_judge(live, verbalise_relation, "relations") if verbose else _quiet(run_judge, live, verbalise_relation, "relations")
    for r in rels:
        r["grounding"]["grounded"] = bool(r["grounding"].get("supported")) and not r["grounding"]["orphaned"]

    graph["grounding_report"] = grounding_report(graph)
    graph["grounding_config"] = {"model": GEMINI_MODEL, "fuzzy_threshold": FUZZY_THRESHOLD,
                                 "judge_partial": JUDGE_PARTIAL, "judge_batch": JUDGE_BATCH}
    out = ROOT / "results" / article_id / TRACK_DIR / OUT_NAME
    out.write_text(json.dumps(graph, ensure_ascii=False, indent=2))
    write_judgment_log(graph, article_id)          # flat per-decision log (JSON + CSV)
    return graph["grounding_report"]

import contextlib, io
def _quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

article_ids = sorted(p.name for p in (ROOT / "results").iterdir()
                     if p.is_dir() and (p / TRACK_DIR / GRAPH_NAME).exists())
print("articles:", article_ids, "\n")

for aid in article_ids:
    try:
        rep = ground_article(aid)
        nd, rl = rep.get("nary_nodes", {}), rep.get("relations", {})
        print(f"{aid:6} nodes {nd.get('grounded')}/{nd.get('total')} ({nd.get('rate')})   "
              f"relations {rl.get('grounded')}/{rl.get('total')} ({rl.get('rate')})   "
              f"failures {rl.get('failures')}")
    except Exception as e:
        print(f"{aid:6} FAILED: {type(e).__name__}: {e}")

articles: ['art1', 'art2', 'art4', 'art5'] 

art1   nodes 44/44 (1.0)   relations 117/133 (0.88)   failures {'intrinsic': 8, 'extrinsic': 8}


KeyboardInterrupt: 